# 06. Assessment 3: Capstone Synthesis Project

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/ACU-ITEC102/blob/main/Week12/06.Assessment-3-Capstone-Project/notebooks/01_06.Assessment-3-Capstone-Project.ipynb)

## Overview & Learning Objectives
This capstone project integrates all key capabilities developed across the **ACU-ITEC102** unit into a unified, professional data science pipeline:
- **Milestone 1**: Ingestion & Structural Quality Auditing.
- **Milestone 2**: Data Wrangling, Type Normalisation & Auxiliary Joins.
- **Milestone 3**: Exploratory Data Analysis & Outlier Detection (Tukey's Fences).
- **Milestone 4**: Visual Storytelling with Matplotlib.
- **Milestone 5**: Privacy-Preserving De-Identification & $k$-Anonymity Assessment.
- **Milestone 6**: Algorithmic Fairness Auditing & Ethical Impact Assessment (EIA).


## Milestone 1: Data Ingestion & Quality Audit
We ingest a simulated raw dataset of recent university graduates that contains text whitespace, varied date formats, string-formatted salaries, and auxiliary lookups.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib

raw_graduates = pd.DataFrame({
    'student_id': [f'ACU-{i:04d}' for i in range(1, 9)],
    'full_name': [
        "  Liam O'CONNOR  ", 'Sophia CHEN', 'ethan smith ', '  Mia AL-MANSOOR',
        'Oliver Taylor', 'Ava WILSON  ', 'Lucas MARTIN', '  Chloe ZHANG'
    ],
    'demographic_cohort': [
        'Group A', 'Group B', 'Group A', 'Group B',
        'Group A', 'Group A', 'Group B', 'Group B'
    ],
    'study_area': [
        'Data Science', 'Software Eng', 'data science', 'Cyber Security',
        'software eng', 'Cyber Security', 'Data Science', 'Software Eng'
    ],
    'grad_date': [
        '2026-06-15', '18/06/2026', '2026/07/01', '05-07-2026',
        '2026.07.12', '15/07/2026', '2026-07-20', '22/07/2026'
    ],
    'gpa': [6.2, 5.8, 4.9, 6.7, 5.4, 6.9, 4.2, 5.9],
    'starting_salary': [
        '$82,000.00 AUD', ' $78500 ', '65000.00', '$94,200.00 AUD',
        '$76,000', ' $220,000.00 AUD ', '$61,500.00 AUD', ' $81,000.00 '
    ],
    'internship_done': ['Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'No', 'Yes'],
    'age': [22, 24, 21, 28, 23, 29, 22, 25],
    'postcode': ['2060', '2060', '3000', '3000', '2060', '4000', '4000', '3000']
})

discipline_lookup = pd.DataFrame({
    'clean_discipline': ['Data Science', 'Software Engineering', 'Cyber Security'],
    'industry_growth_rate': [0.18, 0.14, 0.22]
})

print('Raw Cohort Ingestion:')
display(raw_graduates)
print('\nData Types:')
print(raw_graduates.dtypes)

## Milestone 2: Data Wrangling & Sanitisation Pipeline
We execute our cleaning pipeline:
1. Strip whitespace and title case `full_name`.
2. Map `study_area` to clean, standardised discipline names.
3. Strip regex currency symbols (`$`, commas, `AUD`) and cast to float.
4. Parse mixed date strings using `format='mixed'`.
5. Perform a left join with `discipline_lookup`.

In [ ]:
cleaned = raw_graduates.copy()

# Clean strings & categoricals
cleaned['full_name'] = cleaned['full_name'].str.strip().str.title()
discipline_map = {
    'data science': 'Data Science',
    'software eng': 'Software Engineering',
    'software engineering': 'Software Engineering',
    'cyber security': 'Cyber Security'
}
cleaned['study_area'] = cleaned['study_area'].str.strip().str.lower().map(discipline_map)

# Financial & Datetime cleaning
cleaned['starting_salary'] = (
    cleaned['starting_salary']
    .str.replace(r'[$,AUD\s]', '', regex=True)
    .astype(float)
)
cleaned['grad_date'] = pd.to_datetime(cleaned['grad_date'], format='mixed')

# Left merge lookup table
enriched = pd.merge(
    cleaned,
    discipline_lookup,
    left_on='study_area',
    right_on='clean_discipline',
    how='left'
).drop(columns=['clean_discipline'])

print('Cleaned & Enriched Cohort:')
display(enriched)

## Milestone 3: Exploratory Data Analysis & Outlier Detection
We examine distributions and apply Tukey's Rule on starting salaries:
- $\text{IQR} = Q_3 - Q_1$
- $\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$

In [ ]:
q1 = enriched['starting_salary'].quantile(0.25)
q3 = enriched['starting_salary'].quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f'Salary IQR: ${iqr:,.2f} | Lower Fence: ${lower_fence:,.2f} | Upper Fence: ${upper_fence:,.2f}')
outliers = enriched[(enriched['starting_salary'] < lower_fence) | (enriched['starting_salary'] > upper_fence)]
print(f'Detected {len(outliers)} Outlier(s):')
display(outliers[['student_id', 'full_name', 'study_area', 'starting_salary']])

print('\nMedian Salary by Study Area & Internship Completion:')
display(enriched.groupby(['study_area', 'internship_done'])['starting_salary'].agg(['count', 'median', 'mean']))

## Milestone 4: Visual Storytelling
We design a two-panel publication-quality figure highlighting salary variation by discipline and academic performance.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: Boxplot
disciplines = sorted(enriched['study_area'].unique())
sal_by_disc = [enriched[enriched['study_area'] == d]['starting_salary'] for d in disciplines]
axes[0].boxplot(sal_by_disc, tick_labels=disciplines, patch_artist=True)
axes[0].axhline(upper_fence, color='crimson', linestyle=':', label=f'Outlier Threshold (${upper_fence:,.0f})')
axes[0].set_title('Graduate Starting Salary by Discipline', fontweight='bold')
axes[0].set_ylabel('Starting Salary (AUD)')
axes[0].legend(loc='upper left')
axes[0].grid(True, linestyle='--', alpha=0.5, axis='y')

# Panel 2: Scatter Plot
yes_df = enriched[enriched['internship_done'] == 'Yes']
no_df = enriched[enriched['internship_done'] == 'No']
axes[1].scatter(yes_df['gpa'], yes_df['starting_salary'], color='#2b5c8f', s=80, label='Internship Completed')
axes[1].scatter(no_df['gpa'], no_df['starting_salary'], color='#d95f02', s=80, label='No Internship')
axes[1].set_title('Academic GPA vs Starting Salary', fontweight='bold')
axes[1].set_xlabel('GPA (7-point scale)')
axes[1].set_ylabel('Starting Salary (AUD)')
axes[1].legend(loc='upper left')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Milestone 5: Privacy-Preserving De-Identification & $k$-Anonymity
We prepare the dataset for ethical public research:
1. Pseudonymise `student_id` using salted SHA-256.
2. Drop `full_name`.
3. Coarsen `age` into brackets and `postcode` to 2-digit prefixes.
4. Audit $k$-anonymity across equivalence classes.

In [ ]:
SALT = 'ACU_CAPSTONE_SALT_2026'
anon_df = enriched.copy()

# Salted SHA-256 token
anon_df['anon_id'] = anon_df['student_id'].apply(
    lambda sid: hashlib.sha256(f'{SALT}_{sid}'.encode()).hexdigest()[:8]
)
anon_df = anon_df.drop(columns=['student_id', 'full_name'])

# Coarsen quasi-identifiers
anon_df['age_bracket'] = pd.cut(anon_df['age'], bins=[0, 24, 100], labels=['<=24', '25+']).astype(str)
anon_df['postcode_prefix'] = anon_df['postcode'].astype(str).str[:2] + 'XX'
anon_df = anon_df.drop(columns=['age', 'postcode'])

# Evaluate k-anonymity
equiv = anon_df.groupby(['age_bracket', 'postcode_prefix'], as_index=False).size().rename(columns={'size': 'k_count'})
min_k = equiv['k_count'].min()
print(f'Dataset achieves k-anonymity: k = {min_k}')
display(equiv)
display(anon_df.head())

## Milestone 6: Algorithmic Fairness Audit & Ethical Governance
Suppose an automated scholarship rule requires `gpa >= 5.8`.
We audit whether the model produces adverse impact across demographic cohorts under the 80% rule.

In [ ]:
governance_df = enriched.copy()
governance_df['scholarship_awarded'] = governance_df['gpa'] >= 5.8

rates = governance_df.groupby('demographic_cohort')['scholarship_awarded'].mean()
print('Scholarship Selection Rate by Cohort:')
display(rates)

rate_a = rates.get('Group A', 0.0)
rate_b = rates.get('Group B', 0.0)
dir_score = rate_b / rate_a
print(f'Disparate Impact Ratio (DIR): {dir_score:.3f}')

if dir_score >= 0.80:
    print('✅ [FAIRNESS PASS] Model complies with the 80% adverse impact threshold.')
else:
    print('⚠️ [ALERT] Model violates 80% rule: Threshold mitigation required.')